In [1]:
from datasets import Dataset
import pandas as pd
import numpy as np
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU


In [2]:
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Libraries loaded successfully")
from datasets import load_dataset

dataset = load_dataset(
    "goosmanlei/amazon_reviews_multi",
    "en"
)

print(dataset)

Libraries loaded successfully
DatasetDict({
    train: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
})


In [3]:
def create_sentiment_label(stars):
    if stars <= 3:
        return "negative"
    else:
        return "positive"


def prepare_split(split):
    df = split.to_pandas()
    df["sentiment"] = df["stars"].apply(create_sentiment_label)
    return df


train_df = prepare_split(dataset["train"])
val_df = prepare_split(dataset["validation"])
test_df = prepare_split(dataset["test"])


import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


train_df_clean = train_df.drop_duplicates(
    subset=["review_body", "sentiment"]
).copy()

val_df_clean = val_df.drop_duplicates(
    subset=["review_body", "sentiment"]
).copy()

test_df_clean = test_df.drop_duplicates(
    subset=["review_body", "sentiment"]
).copy()

train_df_clean["text"] = train_df_clean["review_body"].apply(clean_text)
val_df_clean["text"] = val_df_clean["review_body"].apply(clean_text)
test_df_clean["text"] = test_df_clean["review_body"].apply(clean_text)

print("Train:", train_df_clean.shape)
print("Validation:", val_df_clean.shape)
print("Test:", test_df_clean.shape)


Train: (199456, 10)
Validation: (4999, 10)
Test: (5000, 10)


In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "goosmanlei/amazon_reviews_multi",
    "en"
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 200000
    })
    validation: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['review_id', 'product_id', 'reviewer_id', 'stars', 'review_body', 'review_title', 'language', 'product_category'],
        num_rows: 5000
    })
})


In [5]:
def prepare_binary_split(split):
    df = split.to_pandas()

    # 1–3 stars → Negative
    # 4–5 stars → Positive
    df["sentiment"] = df["stars"].apply(
        lambda x: "negative" if x <= 3 else "positive"
    )

    return df


train_df = prepare_binary_split(dataset["train"])
val_df = prepare_binary_split(dataset["validation"])
test_df = prepare_binary_split(dataset["test"])

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 200000
Validation: 5000
Test: 5000


In [6]:
print("Train sentiment distribution:")
print(train_df["sentiment"].value_counts())

print("\nValidation sentiment distribution:")
print(val_df["sentiment"].value_counts())

print("\nTest sentiment distribution:")
print(test_df["sentiment"].value_counts())

Train sentiment distribution:
sentiment
negative    120000
positive     80000
Name: count, dtype: int64

Validation sentiment distribution:
sentiment
negative    3000
positive    2000
Name: count, dtype: int64

Test sentiment distribution:
sentiment
negative    3000
positive    2000
Name: count, dtype: int64


In [7]:
label2id = {
    "negative": 0,
    "positive": 1
}

id2label = {
    0: "negative",
    1: "positive"
}

for df in [train_df, val_df, test_df]:
    df["label"] = df["sentiment"].map(label2id)

print(train_df[["sentiment", "label"]].head())

  sentiment  label
0  negative      0
1  negative      0
2  negative      0
3  negative      0
4  negative      0


In [8]:
classes = np.array([0, 1])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"]
)

print("Class weights:")

for label, weight in zip(classes, class_weights):
    print(f"{id2label[label]}: {weight:.4f}")

Class weights:
negative: 0.8333
positive: 1.2500


In [9]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text


for df in [train_df, val_df, test_df]:
    df["text"] = df["review_body"].apply(clean_text)


print(train_df[["review_body", "text", "label"]].head())

                                         review_body  \
0  Arrived broken. Manufacturer defect. Two of th...   
1  the cabinet dot were all detached from backing...   
2  I received my first order of this product and ...   
3  This product is a piece of shit. Do not buy. D...   
4  went through 3 in one day doesn't fit correct ...   

                                                text  label  
0  arrived broken. manufacturer defect. two of th...      0  
1  the cabinet dot were all detached from backing...      0  
2  i received my first order of this product and ...      0  
3  this product is a piece of shit. do not buy. d...      0  
4  went through 3 in one day doesn't fit correct ...      0  


In [10]:
train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

print(train_dataset)
print(val_dataset)
print(test_dataset)


Dataset({
    features: ['text', 'label'],
    num_rows: 200000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 5000
})
Dataset({
    features: ['text', 'label'],
    num_rows: 5000
})


In [11]:
model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded:", model_name)

Tokenizer loaded: bert-base-uncased


In [12]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )


tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_train)


Map: 100%|██████████| 5000/5000 [00:00<00:00, 10105.19 examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 200000
})


In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

print("Model loaded successfully.")
print("Number of labels:", model.config.num_labels)
print("Labels:", model.config.id2label)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7101.01it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

Model loaded successfully.
Number of labels: 2
Labels: {0: 'negative', 1: 'positive'}


In [14]:
class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float
)

if torch.cuda.is_available():
    class_weights_tensor = class_weights_tensor.to("cuda")

print(class_weights_tensor)

tensor([0.8333, 1.2500], device='cuda:0')


In [15]:
class WeightedTrainer(Trainer):
    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights_tensor
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [16]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")

    return {
        "accuracy": accuracy,
        "f1": f1
    }

In [17]:
training_args = TrainingArguments(
    output_dir="./bert_sentiment_binary_weighted_results",
    num_train_epochs=3,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=2e-5,
    weight_decay=0.01,

    eval_strategy="steps",
    eval_steps=1000,

    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,

    logging_steps=100,

    fp16=True,

    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,

    report_to="none"
)

print("Training arguments configured.")

Training arguments configured.


In [18]:
trainer = WeightedTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,

    processing_class=tokenizer,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("Weighted Trainer is ready.")

Weighted Trainer is ready.


In [19]:
trainer.train()

Step,Training Loss,Validation Loss,Accuracy,F1
1000,1.367737,0.297143,0.885200,0.879861
2000,1.220035,0.280118,0.886800,0.882671
3000,1.201375,0.308582,0.876000,0.872776
4000,1.176648,0.267258,0.896600,0.892119
5000,1.187205,0.274727,0.893600,0.889684
6000,1.120302,0.267392,0.896400,0.892011


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.96it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'ber

TrainOutput(global_step=6000, training_loss=1.2024353357950845, metrics={'train_runtime': 1394.9701, 'train_samples_per_second': 430.117, 'train_steps_per_second': 26.882, 'total_flos': 6314665328640000.0, 'train_loss': 1.2024353357950845, 'epoch': 0.48})

In [20]:
import os

os.makedirs("models/sentiment/bert_binary_weighted", exist_ok=True)

trainer.save_model(
    "models/sentiment/bert_binary_weighted"
)

tokenizer.save_pretrained(
    "models/sentiment/bert_binary_weighted"
)

print("Binary Weighted BERT saved successfully.")

Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it]

Binary Weighted BERT saved successfully.


In [1]:
import os

model_path = "models/sentiment/bert_binary_weighted"

print("Model directory exists:", os.path.exists(model_path))

if os.path.exists(model_path):
    print("\nSaved files:")
    for file in os.listdir(model_path):
        print("-", file)

Model directory exists: True

Saved files:
- config.json
- model.safetensors
- tokenizer.json
- tokenizer_config.json
- training_args.bin


In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "models/sentiment/bert_binary_weighted"

saved_tokenizer = AutoTokenizer.from_pretrained(model_path)

saved_model = AutoModelForSequenceClassification.from_pretrained(
    model_path
)

print("Model loaded successfully.")
print(saved_model.config.id2label)

c:\Users\sondo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4148.12it/s]

Model loaded successfully.
{0: 'negative', 1: 'positive'}


In [4]:
import numpy as np
import torch

from transformers import (
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="macro")

    return {
        "accuracy": accuracy,
        "f1": f1
    }

In [8]:
from datasets import load_dataset, Dataset
import re

dataset = load_dataset(
    "goosmanlei/amazon_reviews_multi",
    "en"
)

test_df = dataset["test"].to_pandas()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

test_df["text"] = test_df["review_body"].apply(clean_text)

test_df["label"] = test_df["stars"].apply(
    lambda x: 0 if x <= 3 else 1
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

print(test_dataset)
print(test_df["label"].value_counts())

Dataset({
    features: ['text', 'label'],
    num_rows: 5000
})
label
0    3000
1    2000
Name: count, dtype: int64


In [11]:
from datasets import load_dataset, Dataset
import re

dataset = load_dataset(
    "goosmanlei/amazon_reviews_multi",
    "en"
)

test_df = dataset["test"].to_pandas()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

test_df["text"] = test_df["review_body"].apply(clean_text)

test_df["label"] = test_df["stars"].apply(
    lambda x: 0 if x <= 3 else 1
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

print(test_dataset)
print(test_df["label"].value_counts())

Dataset({
    features: ['text', 'label'],
    num_rows: 5000
})
label
0    3000
1    2000
Name: count, dtype: int64


In [13]:
def tokenize_function(examples):
    return saved_tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

print(tokenized_test)

Map: 100%|██████████| 5000/5000 [00:00<00:00, 14497.51 examples/s]

Dataset({
    features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 5000
})


In [14]:
test_args = TrainingArguments(
    output_dir="./test_results",
    per_device_eval_batch_size=4,
    report_to="none"
)

test_trainer = Trainer(
    model=saved_model,
    args=test_args,
    processing_class=saved_tokenizer,
    compute_metrics=compute_metrics
)

test_results = test_trainer.evaluate(
    tokenized_test
)

print(test_results)

Training Loss,Validation Loss,Step,Accuracy,F1
No log,0.269292,0,0.887400,0.882580


{'eval_loss': 0.2692924439907074, 'eval_accuracy': 0.8874, 'eval_f1': 0.8825803018299412}


In [15]:
predictions = test_trainer.predict(tokenized_test)

y_true = predictions.label_ids
y_pred = np.argmax(predictions.predictions, axis=-1)

print(classification_report(
    y_true,
    y_pred,
    target_names=["negative", "positive"]
))

              precision    recall  f1-score   support

    negative       0.90      0.91      0.91      3000
    positive       0.86      0.86      0.86      2000

    accuracy                           0.89      5000
   macro avg       0.88      0.88      0.88      5000
weighted avg       0.89      0.89      0.89      5000



In [16]:
cm = confusion_matrix(y_true, y_pred)

print(cm)


[[2725  275]
 [ 288 1712]]
